# TROVE Animal Sentiment Analyzer
## GPU-Accelerated with RoBERTa Transformer Model

This notebook analyzes emotional attitudes towards animals in Australian newspapers using:
- **TROVE API** - National Library of Australia newspaper archives
- **RoBERTa Transformer** - State-of-the-art sentiment analysis (90% accuracy)
- **GPU Acceleration** - 2-3x faster processing on NVIDIA GPUs

### What You'll Get:
- Sentiment scores for each animal
- Word clouds showing common terms
- Sentiment trends over time
- Heat maps by location and date
- Comparative analysis across animals

### Just Run the Cells:
1. **Cell 1**: Automatic installation and GPU setup
2. **Cell 2**: Enter your preferences
3. **Cells 3-6**: Automated analysis and visualizations

**No technical knowledge required - just run each cell!**

In [ ]:
# ========================================
# CELL 1: SMART INSTALLATION & GPU SETUP
# ========================================
# This cell automatically installs missing packages and configures GPU
# Run this cell first - it takes 5-15 minutes on first run

import sys
import subprocess
import importlib.util

def is_package_installed(package_name):
    """Check if a package is already installed."""
    return importlib.util.find_spec(package_name) is not None

def install_if_missing(packages):
    """Install packages only if they're missing."""
    missing = [pkg.split('==')[0] for pkg in packages if not is_package_installed(pkg.split('==')[0].split('[')[0])]
    if missing:
        print(f"Installing {len(missing)} missing packages: {', '.join(missing[:5])}...")
        for pkg in packages:
            if pkg.split('==')[0].split('[')[0] in missing:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print("✓ Installation complete\n")
    else:
        print("✓ All packages already installed\n")

print("="*70)
print("TROVE ANIMAL SENTIMENT ANALYZER - SETUP")
print("="*70 + "\n")

# Check and install PyTorch with CUDA (Windows GPU support)
if not is_package_installed('torch'):
    print("Installing PyTorch with GPU support (this may take 5-10 minutes)...")
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.0.1', 'torchvision==0.15.2', 'torchaudio==2.0.2',
        '--index-url', 'https://download.pytorch.org/whl/cu118'
    ])
    print("✓ PyTorch installed\n")

# Install other required packages
required_packages = [
    'transformers==4.30.2',
    'sentencepiece==0.1.99',
    'pandas',
    'numpy',
    'requests',
    'nltk',
    'matplotlib',
    'seaborn',
    'plotly',
    'wordcloud'
]

install_if_missing(required_packages)

# Import core libraries
import torch
import nltk
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
for dataset in ['stopwords', 'punkt', 'averaged_perceptron_tagger']:
    nltk.download(dataset, quiet=True)

# GPU Detection and Configuration
USE_GPU = torch.cuda.is_available()

if USE_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    # Set optimal batch size based on VRAM
    if gpu_memory >= 20:
        BATCH_SIZE = 32
    elif gpu_memory >= 10:
        BATCH_SIZE = 24
    elif gpu_memory >= 6:
        BATCH_SIZE = 16
    else:
        BATCH_SIZE = 8
    
    # Test GPU
    try:
        test_tensor = torch.rand(5, 3).to('cuda')
        torch.cuda.synchronize()
        GPU_WORKING = True
    except Exception as e:
        GPU_WORKING = False
        USE_GPU = False
        print("⚠ WARNING: GPU detected but not working properly")
        print(f"   Error: {e}")
        print("   Falling back to CPU (slower)\n")
else:
    BATCH_SIZE = 8
    GPU_WORKING = False

# Only report GPU status if there's a problem
if not GPU_WORKING and torch.cuda.is_available():
    print("="*70)
    print("GPU ISSUE DETECTED")
    print("="*70)
    print("GPU was detected but isn't working. Common fixes:")
    print("1. Install Visual C++ Redistributables:")
    print("   https://aka.ms/vs/17/release/vc_redist.x64.exe")
    print("2. Update NVIDIA drivers")
    print("3. Restart Jupyter kernel after installing\n")
    print("Continuing with CPU (slower but functional)...\n")

print("="*70)
print("SETUP COMPLETE")
print("="*70)
if USE_GPU:
    print(f"✓ GPU Acceleration: ENABLED ({gpu_name}, {gpu_memory:.1f}GB)")
    print(f"✓ Optimized batch size: {BATCH_SIZE}")
    print(f"✓ Expected speed: 2-3x faster than CPU")
else:
    print(f"✓ Processing mode: CPU")
    print(f"✓ Batch size: {BATCH_SIZE}")
print("\n→ Ready! Proceed to Cell 2 to configure your analysis")
print("="*70)

In [ ]:
# ========================================
# CELL 2: CONFIGURATION & USER INPUT
# ========================================
# Enter your preferences here

# TROVE API Key (required)
# Get your free key from: https://trove.nla.gov.au/about/create-something/using-api
api_key = input("Enter your TROVE API key: ").strip()

if not api_key:
    print("\n⚠ ERROR: API key is required!")
    print("Get your free key from: https://trove.nla.gov.au/about/create-something/using-api")
    raise ValueError("API key not provided")

print("\n✓ API key accepted\n")

# Animals to analyze
print("Enter animals to analyze (comma-separated):")
print("Example: kangaroo, koala, dingo, platypus, wombat")
animals_input = input("> ").strip()
animals = [animal.strip() for animal in animals_input.split(',') if animal.strip()]

if not animals:
    print("No animals entered. Using defaults: kangaroo, koala, dingo, platypus, wombat")
    animals = ['kangaroo', 'koala', 'dingo', 'platypus', 'wombat']

print(f"\n✓ Will analyze: {', '.join(animals)}\n")

# Date range
print("Enter date range:")
start_year = int(input("  Start year (e.g., 1900): ") or "1900")
end_year = int(input("  End year (e.g., 1950): ") or "1950")
print(f"\n✓ Date range: {start_year} to {end_year}\n")

# Location selection
print("Location filter options:")
print("1. All of Australia (no filter)")
print("2. Specific state(s)")
print("3. Specific newspaper(s)")
location_choice = input("Choose option (1/2/3): ").strip()

states = []
newspapers = []

if location_choice == '2':
    print("\nEnter state codes (comma-separated):")
    print("Options: NSW, VIC, QLD, SA, WA, TAS, NT, ACT")
    states_input = input("> ").strip()
    states = [state.strip().upper() for state in states_input.split(',') if state.strip()]
    print(f"✓ Filtering by states: {', '.join(states)}\n")
elif location_choice == '3':
    print("\nEnter newspaper IDs (comma-separated):")
    print("Example: 11, 35, 166 (find IDs at trove.nla.gov.au)")
    newspapers_input = input("> ").strip()
    newspapers = [n.strip() for n in newspapers_input.split(',') if n.strip()]
    print(f"✓ Filtering by {len(newspapers)} newspaper(s)\n")
else:
    print("✓ Searching all of Australia\n")

# Number of articles
print("How many articles per animal?")
print("Recommended: 50-200 for quick analysis, 500+ for comprehensive")
max_articles = int(input("Articles per animal (default 100): ") or "100")
print(f"\n✓ Will retrieve up to {max_articles} articles per animal\n")

# Summary
print("="*70)
print("CONFIGURATION SUMMARY")
print("="*70)
print(f"Animals: {', '.join(animals)}")
print(f"Date range: {start_year}-{end_year}")
if states:
    print(f"States: {', '.join(states)}")
elif newspapers:
    print(f"Newspapers: {len(newspapers)} selected")
else:
    print(f"Location: All of Australia")
print(f"Articles per animal: {max_articles}")
print(f"Total articles to retrieve: ~{len(animals) * max_articles}")
print("="*70)
print("\n✓ Configuration complete! Proceed to Cell 3 to start analysis")

In [ ]:
# ========================================
# CELL 3: LOAD ROBERTA MODEL
# ========================================
# Loading RoBERTa sentiment analysis model

print("="*70)
print("LOADING ROBERTA TRANSFORMER MODEL")
print("="*70)
print("\nLoading RoBERTa model (first time may take 1-2 minutes to download)...\n")

from transformers import pipeline
import os

# Suppress transformer warnings
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Load RoBERTa sentiment model
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 if USE_GPU else -1,
    torch_dtype=torch.float16 if USE_GPU else torch.float32,
    truncation=True,
    max_length=512
)

# Test the model
test_result = sentiment_pipeline("The kangaroo is a wonderful animal.")[0]

print("="*70)
print("MODEL LOADED SUCCESSFULLY")
print("="*70)
print(f"Model: RoBERTa (90% accuracy on sentiment tasks)")
print(f"Device: {'GPU' if USE_GPU else 'CPU'}")
if USE_GPU:
    print(f"Precision: FP16 (mixed precision for speed)")
print(f"Test: '{test_result['label']}' (score: {test_result['score']:.3f})")
print("="*70)
print("\n✓ Ready for sentiment analysis! Proceed to Cell 4")

In [ ]:
# ========================================
# CELL 4: HELPER FUNCTIONS
# ========================================
# Define all required functions

import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime
from collections import Counter
import re

def query_trove_animal(animal, api_key, start_year, end_year, states=None, newspapers=None, max_results=100):
    """Query TROVE API for articles mentioning a specific animal."""
    params = {
        'key': api_key,
        'zone': 'newspaper',
        'include': 'articleText',
        'n': 100,
        'encoding': 'json',
        'bulkHarvest': 'false',
        'reclevel': 'brief',
        'sortby': 'relevance'
    }
    
    if states:
        params['l-state'] = states
    if newspapers:
        params['l-title'] = newspapers
    
    params['q'] = f'{animal} date:[{start_year} TO {end_year}]'
    
    all_articles = []
    total_retrieved = 0
    
    print(f"Querying TROVE for '{animal}'...", end=" ", flush=True)
    
    response = requests.get('https://api.trove.nla.gov.au/v2/result', params=params)
    
    if response.status_code != 200:
        print(f"Error: API returned status code {response.status_code}")
        return pd.DataFrame()
    
    data = response.json()
    
    try:
        total_available = int(data['response']['zone'][0]['records']['total'])
        articles = data['response']['zone'][0]['records'].get('article', [])
    except (KeyError, IndexError):
        print("No results found.")
        return pd.DataFrame()
    
    all_articles.extend(articles)
    total_retrieved = len(articles)
    
    # Retrieve additional pages if needed
    while total_retrieved < min(max_results, total_available):
        params['s'] = f"*:{total_retrieved}"
        time.sleep(0.2)  # Rate limiting
        
        response = requests.get('https://api.trove.nla.gov.au/v2/result', params=params)
        if response.status_code != 200:
            break
            
        data = response.json()
        try:
            articles = data['response']['zone'][0]['records'].get('article', [])
            if not articles:
                break
            all_articles.extend(articles)
            total_retrieved += len(articles)
        except (KeyError, IndexError):
            break
    
    print(f"Retrieved {total_retrieved} articles")
    
    if not all_articles:
        return pd.DataFrame()
    
    df = pd.json_normalize(all_articles)
    df['animal'] = animal
    
    # Clean article text
    if 'articleText' in df.columns:
        df['article_text'] = df['articleText'].str.replace(r'<[^<>]*>', '', regex=True)
    
    # Parse dates
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
    
    return df


def batch_analyze_sentiment(texts, batch_size=None):
    """Analyze sentiment for multiple texts using RoBERTa."""
    if batch_size is None:
        batch_size = BATCH_SIZE
    
    scores = []
    start_time = time.time()
    
    print(f"Analyzing {len(texts)} articles with RoBERTa...")
    print(f"Batch size: {batch_size} | Device: {'GPU' if USE_GPU else 'CPU'}")
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch = [str(text)[:2048] for text in batch]  # Truncate long texts
        
        try:
            results = sentiment_pipeline(batch, truncation=True, max_length=512)
            
            for result in results:
                label = result['label'].upper()
                score = result['score']
                
                # Convert to -1 to +1 scale
                if 'NEGATIVE' in label or 'LABEL_0' in label:
                    scores.append(-score)
                elif 'POSITIVE' in label or 'LABEL_2' in label:
                    scores.append(score)
                else:
                    scores.append(0)
        except Exception as e:
            print(f"\nError in batch {i}: {e}")
            scores.extend([0] * len(batch))
        
        # Progress indicator
        if (i + batch_size) % 50 == 0 or (i + batch_size) >= len(texts):
            elapsed = time.time() - start_time
            progress = min(100, int((i + batch_size) / len(texts) * 100))
            articles_per_sec = (i + batch_size) / elapsed if elapsed > 0 else 0
            eta = (len(texts) - (i + batch_size)) / articles_per_sec if articles_per_sec > 0 else 0
            
            print(f"\rProgress: {progress}% ({min(i + batch_size, len(texts))}/{len(texts)}) | "
                  f"{articles_per_sec:.1f} articles/sec | ETA: {eta/60:.1f}min", end="", flush=True)
        
        # GPU cache management
        if USE_GPU and i > 0 and i % 100 == 0:
            torch.cuda.empty_cache()
    
    elapsed = time.time() - start_time
    print(f"\n✓ Completed in {elapsed/60:.1f} minutes ({len(texts)/elapsed:.2f} articles/sec)")
    
    if USE_GPU:
        peak_vram = torch.cuda.max_memory_allocated(0) / 1024**3
        print(f"  Peak VRAM usage: {peak_vram:.2f} GB")
        torch.cuda.empty_cache()
    
    return scores


def categorize_sentiment(score):
    """Categorize sentiment score."""
    if score > 0.1:
        return 'positive'
    elif score < -0.1:
        return 'negative'
    else:
        return 'neutral'


print("✓ Helper functions loaded")
print("\n→ Proceed to Cell 5 to retrieve and analyze articles")

In [ ]:
# ========================================
# CELL 5: RETRIEVE & ANALYZE
# ========================================
# Retrieve articles from TROVE and perform sentiment analysis

print("="*70)
print("RETRIEVING ARTICLES FROM TROVE")
print("="*70 + "\n")

# Collect data for all animals
all_data = []

for animal in animals:
    df = query_trove_animal(
        animal=animal,
        api_key=api_key,
        start_year=start_year,
        end_year=end_year,
        states=states if states else None,
        newspapers=newspapers if newspapers else None,
        max_results=max_articles
    )
    
    if not df.empty:
        all_data.append(df)
    
    time.sleep(0.5)  # Be nice to TROVE API

if not all_data:
    print("\n⚠ No articles retrieved. Check your API key and parameters.")
    raise ValueError("No data collected from TROVE")

# Combine all data
df_animals = pd.concat(all_data, ignore_index=True)

print(f"\n✓ Total articles collected: {len(df_animals)}")
print("\nArticles per animal:")
print(df_animals.groupby('animal').size())

# Perform sentiment analysis
print("\n" + "="*70)
print("PERFORMING SENTIMENT ANALYSIS")
print("="*70 + "\n")

if 'article_text' in df_animals.columns:
    texts = df_animals['article_text'].fillna('').astype(str).tolist()
    sentiment_scores = batch_analyze_sentiment(texts)
    
    df_animals['sentiment_score'] = sentiment_scores
    df_animals['sentiment_category'] = df_animals['sentiment_score'].apply(categorize_sentiment)
    
    print("\n" + "="*70)
    print("SENTIMENT ANALYSIS RESULTS")
    print("="*70 + "\n")
    
    # Calculate average sentiment by animal
    sentiment_by_animal = df_animals.groupby('animal')['sentiment_score'].mean().sort_values(ascending=False)
    
    print("Average sentiment by animal:")
    print("-" * 50)
    for animal, score in sentiment_by_animal.items():
        sentiment = "POSITIVE" if score > 0.1 else "NEGATIVE" if score < -0.1 else "NEUTRAL"
        bar = '█' * int(abs(score) * 50)
        print(f"{animal.capitalize():15s}: {score:+.3f} [{sentiment:8s}] {bar}")
    
    print("\n" + "="*70)
    print("✓ Analysis complete! Proceed to Cell 6 for visualizations")
    print("="*70)
else:
    print("⚠ No article text found in results")

In [ ]:
# ========================================
# CELL 6: VISUALIZATIONS
# ========================================
# Generate word clouds, graphs, and heat maps

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from nltk.corpus import stopwords

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*70)
print("GENERATING VISUALIZATIONS")
print("="*70 + "\n")

# Prepare stopwords
stop_words = set(stopwords.words('english'))
stop_words.update(['said', 'would', 'also', 'one', 'two', 'mr', 'mrs', 'will', 'may'])

# ============================================================
# 1. WORD CLOUDS FOR EACH ANIMAL
# ============================================================
print("1. Generating word clouds...\n")

num_animals = len(animals)
cols = min(3, num_animals)
rows = (num_animals + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
if num_animals == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for idx, animal in enumerate(animals):
    animal_texts = ' '.join(df_animals[df_animals['animal'] == animal]['article_text'].fillna(''))
    
    if animal_texts.strip():
        wordcloud = WordCloud(
            width=800, height=400,
            background_color='white',
            stopwords=stop_words,
            max_words=100,
            colormap='viridis'
        ).generate(animal_texts)
        
        axes[idx].imshow(wordcloud, interpolation='bilinear')
        axes[idx].set_title(f'{animal.capitalize()} - Word Cloud', fontsize=14, fontweight='bold')
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, f'No data for {animal}', ha='center', va='center')
        axes[idx].axis('off')

# Hide unused subplots
for idx in range(num_animals, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(f'wordclouds_{start_year}_{end_year}.png', dpi=300, bbox_inches='tight')
plt.show()
print("  ✓ Word clouds saved as 'wordclouds_{start_year}_{end_year}.png'\n")

# ============================================================
# 2. SENTIMENT COMPARISON BAR CHART
# ============================================================
print("2. Creating sentiment comparison chart...\n")

sentiment_summary = df_animals.groupby('animal')['sentiment_score'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['red' if x < -0.1 else 'green' if x > 0.1 else 'gray' for x in sentiment_summary.values]
sentiment_summary.plot(kind='barh', ax=ax, color=colors)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Average Sentiment Score', fontsize=12)
ax.set_ylabel('Animal', fontsize=12)
ax.set_title(f'Animal Sentiment Comparison ({start_year}-{end_year})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'sentiment_comparison_{start_year}_{end_year}.png', dpi=300, bbox_inches='tight')
plt.show()
print("  ✓ Sentiment comparison saved as 'sentiment_comparison_{start_year}_{end_year}.png'\n")

# ============================================================
# 3. SENTIMENT TRENDS OVER TIME
# ============================================================
print("3. Creating sentiment trends over time...\n")

if 'year' in df_animals.columns:
    yearly_sentiment = df_animals.groupby(['year', 'animal'])['sentiment_score'].mean().reset_index()
    
    fig = px.line(
        yearly_sentiment,
        x='year',
        y='sentiment_score',
        color='animal',
        markers=True,
        title=f'Sentiment Trends Over Time ({start_year}-{end_year})',
        labels={'sentiment_score': 'Average Sentiment', 'year': 'Year', 'animal': 'Animal'}
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
    fig.update_layout(height=500, hovermode='x unified')
    fig.write_html(f'sentiment_trends_{start_year}_{end_year}.html')
    fig.show()
    print("  ✓ Interactive trend chart saved as 'sentiment_trends_{start_year}_{end_year}.html'\n")

# ============================================================
# 4. HEAT MAP - SENTIMENT BY YEAR AND ANIMAL
# ============================================================
print("4. Creating heat map...\n")

if 'year' in df_animals.columns:
    # Create pivot table
    heatmap_data = df_animals.pivot_table(
        values='sentiment_score',
        index='animal',
        columns='year',
        aggfunc='mean'
    )
    
    # Matplotlib heatmap
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(
        heatmap_data,
        annot=False,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        cbar_kws={'label': 'Sentiment Score'},
        ax=ax
    )
    ax.set_title(f'Sentiment Heat Map by Year and Animal ({start_year}-{end_year})', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Animal', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'sentiment_heatmap_{start_year}_{end_year}.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("  ✓ Heat map saved as 'sentiment_heatmap_{start_year}_{end_year}.png'\n")

# ============================================================
# 5. SENTIMENT DISTRIBUTION
# ============================================================
print("5. Creating sentiment distribution charts...\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for animal in animals:
    animal_data = df_animals[df_animals['animal'] == animal]['sentiment_score']
    axes[0].hist(animal_data, alpha=0.6, bins=20, label=animal.capitalize())

axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Sentiment Score', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Sentiment Score Distribution', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
df_animals.boxplot(column='sentiment_score', by='animal', ax=axes[1])
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Animal', fontsize=11)
axes[1].set_ylabel('Sentiment Score', fontsize=11)
axes[1].set_title('Sentiment Score Distribution by Animal', fontsize=12, fontweight='bold')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig(f'sentiment_distribution_{start_year}_{end_year}.png', dpi=300, bbox_inches='tight')
plt.show()
print("  ✓ Distribution charts saved as 'sentiment_distribution_{start_year}_{end_year}.png'\n")

# ============================================================
# 6. SAVE RESULTS TO CSV
# ============================================================
print("6. Saving data to CSV...\n")

output_file = f'animal_sentiment_results_{start_year}_{end_year}.csv'
df_animals.to_csv(output_file, index=False)
print(f"  ✓ Full dataset saved as '{output_file}'")

# Save summary statistics
summary = df_animals.groupby('animal').agg({
    'sentiment_score': ['mean', 'std', 'min', 'max', 'count'],
    'sentiment_category': lambda x: (x == 'positive').sum(),
}).round(4)

summary.columns = ['Mean', 'Std Dev', 'Min', 'Max', 'Article Count', 'Positive Articles']
summary_file = f'sentiment_summary_{start_year}_{end_year}.csv'
summary.to_csv(summary_file)
print(f"  ✓ Summary statistics saved as '{summary_file}'\n")

print("="*70)
print("ALL VISUALIZATIONS COMPLETE")
print("="*70)
print("\nGenerated files:")
print(f"  - wordclouds_{start_year}_{end_year}.png")
print(f"  - sentiment_comparison_{start_year}_{end_year}.png")
print(f"  - sentiment_trends_{start_year}_{end_year}.html (interactive)")
print(f"  - sentiment_heatmap_{start_year}_{end_year}.png")
print(f"  - sentiment_distribution_{start_year}_{end_year}.png")
print(f"  - {output_file} (full dataset)")
print(f"  - {summary_file} (summary statistics)")
print("\n✓ Analysis complete!")
print("="*70)